# Atividade 2 – Parte 3: Introdução à Segmentação Semântica e por Instâncias
**TECA2 – Tópicos em Visão Computacional**
**Alunos:** Henryque Oliveira, Matheus Marinho e Rodrigo Oliveira

Demonstração guiada com modelos pré-treinados, contrastando o pipeline clássico
das Partes 1 e 2 com a abordagem por aprendizado profundo (Corke, cap. 12.1.1.3 e 12.1.4).

**Pipeline:**
1. **Etapa 1** – Escolha da imagem
2. **Etapa 2** – Segmentação **semântica** com DeepLabV3-ResNet50 (treinado em Pascal VOC)
3. **Etapa 3** – Segmentação **por instâncias** com Mask R-CNN-ResNet50-FPN (treinado em COCO)
4. **Etapa 4** – Discussão conceitual: semântica × instâncias × pipeline clássico


## 0. Importações e Configurações

In [ ]:
import os
import time
import numpy as np
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import torch
from torchvision.io import read_image
from torchvision.models.segmentation import (
    deeplabv3_resnet50, DeepLabV3_ResNet50_Weights
)
from torchvision.models.detection import (
    maskrcnn_resnet50_fpn, MaskRCNN_ResNet50_FPN_Weights
)

os.makedirs('../image/output', exist_ok=True)
plt.rcParams['figure.dpi'] = 110

IMG_PATH = '../image/input/cena_dl.jpg'
DEVICE = torch.device('cpu')
torch.manual_seed(0)

print(f'torch={torch.__version__}  device={DEVICE}')
print(f'Imagem de entrada: {IMG_PATH}')

---
## Etapa 1 – Escolha da Imagem

**Imagem utilizada:** `cena_dl.jpg` — duas instâncias da mesma classe (gatos) repousando sobre um sofá,
com dois controles remotos sobre a cama. A escolha é proposital: a coexistência de **dois objetos da
mesma classe** torna evidente a diferença entre rotular pixels por classe (semântica) e separar
indivíduos (instâncias).

**Fonte:** COCO val2017 — `000000039769.jpg` (CC BY 4.0, [cocodataset.org](https://cocodataset.org)).
Trata-se de uma imagem amplamente utilizada como exemplo canônico em tutoriais de visão computacional
(Hugging Face, torchvision).

**Classes esperadas (Pascal VOC / COCO):** `cat`, `sofa/couch`, `remote`.

In [ ]:
img_bgr = cv2.imread(IMG_PATH)
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
H, W = img_rgb.shape[:2]

print(f'Resolução: {W} × {H}  ({img_rgb.shape[2]} canais)')

plt.figure(figsize=(7, 5))
plt.imshow(img_rgb)
plt.title('Imagem de entrada (Etapa 1)', fontsize=12)
plt.axis('off')
plt.tight_layout()
plt.savefig('../image/output/dl_etapa1_entrada.png', bbox_inches='tight')
plt.show()

---
## Etapa 2 – Segmentação Semântica (DeepLabV3-ResNet50)

A **segmentação semântica** atribui uma classe a cada pixel. Aqui usamos `DeepLabV3-ResNet50` da
torchvision com pesos `COCO_WITH_VOC_LABELS_V1`, que cobrem as **21 classes do Pascal VOC** (incluindo
*cat*, *sofa*, *person*, *dog*, *car* etc.).

Diferença para o pipeline clássico das Partes 1 e 2: não definimos limiar de cinza, conectividade
ou descritor geométrico — a rede aprendeu a mapear pixels → classes a partir de imagens anotadas.

In [ ]:
# Carrega DeepLabV3 + transforms da torchvision (já incluem normalização ImageNet)
weights_seg = DeepLabV3_ResNet50_Weights.COCO_WITH_VOC_LABELS_V1
print('Carregando DeepLabV3-ResNet50 (Pascal VOC)...')
t0 = time.time()
model_seg = deeplabv3_resnet50(weights=weights_seg).eval().to(DEVICE)
preprocess_seg = weights_seg.transforms()
voc_classes = weights_seg.meta['categories']
print(f'  pronto em {time.time()-t0:.1f}s | {len(voc_classes)} classes')

# Inferência
img_tensor = read_image(IMG_PATH)              # uint8 CHW
batch = preprocess_seg(img_tensor).unsqueeze(0).to(DEVICE)
print('Inferência...')
t0 = time.time()
with torch.no_grad():
    logits = model_seg(batch)['out'][0]        # [21, h, w]
print(f'  pronto em {time.time()-t0:.2f}s | logits {tuple(logits.shape)}')

# argmax → mapa de classes (HxW)
pred_seg = logits.argmax(0).cpu().numpy().astype(np.int32)

# Redimensiona para a resolução original da imagem (DeepLabV3 trabalha em ~520 px)
pred_seg = cv2.resize(pred_seg.astype(np.uint8), (W, H), interpolation=cv2.INTER_NEAREST)

classes_presentes = sorted(set(np.unique(pred_seg).tolist()))
print('\nClasses detectadas na imagem:')
for c in classes_presentes:
    pct = 100 * (pred_seg == c).sum() / pred_seg.size
    print(f'  [{c:2d}] {voc_classes[c]:15s}  {pct:5.1f}% dos pixels')

In [ ]:
# Constrói paleta de cores VOC e mapa colorido
def paleta_voc(n_classes=21):
    """Paleta padrão do Pascal VOC (mesmas cores usadas em literatura)."""
    palette = np.zeros((n_classes, 3), dtype=np.uint8)
    for j in range(n_classes):
        lab, r, g, b = j, 0, 0, 0
        for i in range(8):
            r |= ((lab >> 0) & 1) << (7 - i)
            g |= ((lab >> 1) & 1) << (7 - i)
            b |= ((lab >> 2) & 1) << (7 - i)
            lab >>= 3
        palette[j] = [r, g, b]
    return palette

PALETA = paleta_voc(len(voc_classes))
mapa_colorido = PALETA[pred_seg]                              # HxWx3 uint8
overlay = cv2.addWeighted(img_rgb, 0.55, mapa_colorido, 0.45, 0)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
axes[0].imshow(img_rgb);          axes[0].set_title('Imagem original');               axes[0].axis('off')
axes[1].imshow(mapa_colorido);    axes[1].set_title('Mapa de segmentação semântica'); axes[1].axis('off')
axes[2].imshow(overlay);          axes[2].set_title('Sobreposição (α=0.45)');         axes[2].axis('off')

# Legenda apenas com as classes presentes (exceto fundo)
patches = [
    mpatches.Patch(color=PALETA[c] / 255, label=f'{voc_classes[c]} ({100*(pred_seg==c).sum()/pred_seg.size:.1f}%)')
    for c in classes_presentes
]
fig.legend(handles=patches, loc='lower center',
           bbox_to_anchor=(0.5, -0.02), ncol=len(patches), fontsize=10, frameon=False)
plt.suptitle('Etapa 2 – Segmentação Semântica (DeepLabV3-ResNet50, Pascal VOC)', fontsize=13)
plt.tight_layout()
plt.savefig('../image/output/dl_etapa2_semantica.png', bbox_inches='tight')
plt.show()

---
## Etapa 3 – Segmentação por Instâncias (Mask R-CNN-ResNet50-FPN)

A **segmentação por instâncias** vai além: além da classe, separa cada objeto individual
(*cat 1*, *cat 2*, *remote 1*, *remote 2*…). Usamos `Mask R-CNN ResNet50-FPN` com pesos `COCO_V1`,
treinados em **80 classes do COCO**.

Cada detecção retorna: classe, *score* de confiança, *bounding box* e uma **máscara binária** própria.

In [ ]:
SCORE_THR = 0.7   # mantém apenas detecções confiáveis
MASK_THR  = 0.5

weights_inst = MaskRCNN_ResNet50_FPN_Weights.COCO_V1
print('Carregando Mask R-CNN-ResNet50-FPN (COCO)...')
t0 = time.time()
model_inst = maskrcnn_resnet50_fpn(weights=weights_inst).eval().to(DEVICE)
preprocess_inst = weights_inst.transforms()
coco_classes = weights_inst.meta['categories']
print(f'  pronto em {time.time()-t0:.1f}s | {len(coco_classes)} classes')

batch_inst = preprocess_inst(img_tensor).unsqueeze(0).to(DEVICE)
print('Inferência...')
t0 = time.time()
with torch.no_grad():
    out_inst = model_inst(batch_inst)[0]
print(f'  pronto em {time.time()-t0:.2f}s')

scores = out_inst['scores'].cpu().numpy()
keep = scores >= SCORE_THR
masks  = out_inst['masks' ][keep, 0].cpu().numpy()        # (N, H, W) prob.
boxes  = out_inst['boxes' ][keep    ].cpu().numpy()       # (N, 4)
labels = out_inst['labels'][keep    ].cpu().numpy()       # (N,)
scores = scores[keep]

print(f'\n{len(masks)} instâncias com score ≥ {SCORE_THR}:')
contagem = {}
for i, (lbl, sc) in enumerate(zip(labels, scores), start=1):
    cls = coco_classes[int(lbl)]
    contagem[cls] = contagem.get(cls, 0) + 1
    print(f'  inst #{i:2d} | {cls:15s} | score={sc:.3f}')
print('\nContagem por classe:', contagem)

In [ ]:
# Visualiza cada máscara de instância em painel separado
n = len(masks)
ncols = min(n, 4)
nrows = (n + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 3.2 * nrows))
axes = np.atleast_2d(axes)

for ax in axes.ravel():
    ax.axis('off')

for i, (mask, box, lbl, sc) in enumerate(zip(masks, boxes, labels, scores)):
    ax = axes.ravel()[i]
    bin_mask = (mask >= MASK_THR).astype(np.uint8)
    # Crop com folga em torno da bbox
    x1, y1, x2, y2 = box.astype(int)
    pad = 15
    x1, y1 = max(x1 - pad, 0), max(y1 - pad, 0)
    x2, y2 = min(x2 + pad, W), min(y2 + pad, H)
    rgb_crop  = img_rgb[y1:y2, x1:x2].copy()
    mask_crop = bin_mask[y1:y2, x1:x2]

    # Aplica máscara em destaque (escurece o fundo fora do objeto)
    rgb_crop[mask_crop == 0] = (rgb_crop[mask_crop == 0] * 0.30).astype(np.uint8)

    ax.imshow(rgb_crop)
    ax.set_title(f'Inst. #{i+1} – {coco_classes[int(lbl)]} (score={sc:.2f})', fontsize=10)

plt.suptitle(f'Etapa 3 – Máscaras individuais por instância ({n} objetos)', fontsize=12)
plt.tight_layout()
plt.savefig('../image/output/dl_etapa3_mascaras.png', bbox_inches='tight')
plt.show()

In [ ]:
# Imagem composta: todas as instâncias coloridas + contornos + ID/classe
def cor_instancia(i, n):
    return tuple(int(c * 255) for c in plt.get_cmap('tab10', max(n, 1))(i)[:3])

img_inst_overlay = img_rgb.copy().astype(np.float32)
img_contours     = img_rgb.copy()

alpha = 0.45
for i, (mask, box, lbl, sc) in enumerate(zip(masks, boxes, labels, scores)):
    cor = cor_instancia(i, len(masks))
    bin_mask = (mask >= MASK_THR).astype(np.uint8)

    color_layer = np.zeros_like(img_rgb, dtype=np.float32)
    color_layer[bin_mask == 1] = cor
    img_inst_overlay = np.where(bin_mask[..., None] == 1,
                                (1 - alpha) * img_inst_overlay + alpha * color_layer,
                                img_inst_overlay)

    cnts, _ = cv2.findContours(bin_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cv2.drawContours(img_contours, cnts, -1, cor, 2)

    x1, y1, x2, y2 = box.astype(int)
    cv2.rectangle(img_contours, (x1, y1), (x2, y2), cor, 2)
    label_txt = f'#{i+1} {coco_classes[int(lbl)]} {sc:.2f}'
    cv2.putText(img_contours, label_txt, (x1, max(y1 - 6, 14)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.55, cor, 2)

img_inst_overlay = img_inst_overlay.clip(0, 255).astype(np.uint8)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].imshow(img_inst_overlay); axes[0].set_title('Máscaras coloridas por instância'); axes[0].axis('off')
axes[1].imshow(img_contours);     axes[1].set_title('Contornos + bounding boxes + classes'); axes[1].axis('off')
plt.suptitle('Etapa 3 – Visão composta de todas as instâncias detectadas', fontsize=13)
plt.tight_layout()
plt.savefig('../image/output/dl_etapa3_composta.png', bbox_inches='tight')
plt.show()

---
## Etapa 4 – Comparação Visual: Semântica × Instâncias

A semântica pinta com **uma única cor** todos os pixels da classe `cat`; a segmentação por
instâncias atribui **uma cor por indivíduo**, separando os dois gatos mesmo que pertençam à
mesma classe.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

axes[0].imshow(img_rgb)
axes[0].set_title('Original')
axes[0].axis('off')

axes[1].imshow(cv2.addWeighted(img_rgb, 0.45, mapa_colorido, 0.55, 0))
axes[1].set_title('Semântica: 1 cor por CLASSE')
axes[1].axis('off')

axes[2].imshow(img_inst_overlay)
axes[2].set_title('Instâncias: 1 cor por OBJETO')
axes[2].axis('off')

plt.suptitle('Etapa 4 – Comparação visual entre as duas abordagens', fontsize=13)
plt.tight_layout()
plt.savefig('../image/output/dl_etapa4_comparacao.png', bbox_inches='tight')
plt.show()

# Resumo numérico das duas abordagens
print('=' * 60)
print('Resumo numérico')
print('=' * 60)
print(f'Semântica  (DeepLabV3 / Pascal VOC): {len(classes_presentes)} classes presentes,')
print(f'                                     {", ".join(voc_classes[c] for c in classes_presentes)}')
print(f'Instâncias (Mask R-CNN / COCO):      {len(masks)} objetos detectados,')
for cls, n_obj in contagem.items():
    print(f'                                     • {n_obj} × {cls}')

---
## Discussão Conceitual

### Segmentação semântica × pipeline clássico (Partes 1 e 2)

| Aspecto | Pipeline clássico (Partes 1 e 2) | Segmentação semântica (DeepLabV3) |
|---|---|---|
| **Como decide o que é objeto** | Limiar de cinza (Otsu), morfologia, conectividade | Rede neural treinada em milhares de imagens anotadas |
| **Saída por pixel** | Binário (objeto / fundo) | Classe entre as 21 do Pascal VOC |
| **Funciona em cena qualquer?** | Apenas em cenários controlados (fundo uniforme) | Sim, dentro do conjunto de classes treinadas |
| **Interpretabilidade** | Alta — cada parâmetro tem significado claro | Baixa — saída é um vetor de pesos aprendidos |
| **Custo de computação** | Milissegundos em CPU | Segundos em CPU (~2 s na imagem usada) |
| **Pré-requisito** | Bom contraste objeto / fundo | Modelo pré-treinado e GPU/CPU recente |

### Segmentação por instâncias × semântica

A grande vantagem da segmentação por instâncias é **separar indivíduos da mesma classe**.
No nosso experimento, a semântica reconheceu “cat” como uma única região contígua;
o Mask R-CNN identificou **dois gatos distintos** (`cat #1` e `cat #2`) e ainda detectou
controles remotos individuais e a cama, com *bounding box* e máscara próprias para cada um.
É a abordagem natural para **contagem de pessoas em multidão**, **rastreamento individual**
em vídeo, ou **inspeção peça a peça** em linhas de produção.

### O que foi bem reconhecido no experimento

- **Dois gatos** corretamente separados como instâncias distintas (scores > 0,98).
- **Controles remotos** detectados com bordas justas — algo que o pipeline clássico das
  Partes 1 e 2 não conseguiria sem regras manuais sob mesmo fundo escuro.
- A semântica concordou na presença de `cat` e `sofa`; os controles, por não serem classe
  do Pascal VOC, ficaram absorvidos no fundo (`__background__`).

### Limitações observadas

- **Vocabulário fixo**: o Pascal VOC não tem `remote` nem `bed`, então a semântica não os
  rotula. O COCO tem ambos — daí a riqueza maior na detecção por instâncias.
- **Bordas imprecisas em regiões de toque**: onde os gatos encostam um no outro a máscara
  pode “vazar” entre instâncias.
- **Classes ausentes do conjunto de treino** ficam invisíveis a essas redes; um pipeline
  clássico binarizado, em contraste, segmentaria qualquer região de bom contraste
  (mas sem nomeá-la).
- **Custo computacional** em CPU é alto comparado ao pipeline clássico — em produção, esses
  modelos são tipicamente servidos em GPU ou em variantes mais leves (MobileNet, YOLO-seg).

### Aplicações vantajosas para essa abordagem

- **Direção autônoma**: segmentar pessoas, carros e ciclistas em tempo real.
- **Imagem médica**: separar tumores, órgãos, lesões individuais (Mask R-CNN é base de
  muitos pipelines em radiologia).
- **Agricultura de precisão / contagem em multidão**: contar instâncias da mesma espécie.
- **Robótica / *picking* industrial**: identificar e separar peças sobrepostas.

### Conclusão

A Parte 3 deixa claro que o pipeline clássico das Partes 1 e 2 e a abordagem por DL **não são
substitutos diretos**: o clássico domina em cenários controlados, é interpretável e barato;
a abordagem por DL generaliza para cenas reais e abre a possibilidade de **descrição
semântica** dos objetos — algo que, no clássico, exigiria um classificador adicional sobre
as *features* da Parte 2.
